<style>
.jp-RenderedHTMLCommon { font-size: 22px !important; line-height: 1.8 !important; }
.jp-RenderedHTMLCommon h1 { font-size: 2.2em !important; }
.jp-RenderedHTMLCommon h2 {
  font-size: 1.7em !important;
  border-left: 5px solid #ffb300 !important;
  padding: 10px 0 10px 18px !important;
  margin-top: 40px !important;
  margin-bottom: 16px !important;
  background: linear-gradient(90deg, rgba(255,179,0,0.08) 0%, transparent 60%) !important;
  border-radius: 4px !important;
}
.jp-RenderedHTMLCommon h3 {
  font-size: 1.35em !important;
  border-bottom: 2px solid rgba(255,179,0,0.3) !important;
  padding-bottom: 8px !important;
  margin-top: 30px !important;
  margin-bottom: 12px !important;
  color: #ffb300 !important;
}
.jp-RenderedHTMLCommon h4 {
  font-size: 1.15em !important;
  font-style: italic !important;
  color: #ffb300 !important;
  margin-top: 22px !important;
}
</style>

<div style="background: linear-gradient(135deg, #E65100 0%, #FF8F00 50%, #FFCA28 100%); padding: 40px 45px; border-radius: 16px; margin-bottom: 30px; border: 1px solid rgba(255,179,0,0.15); max-width: 95%; overflow: hidden; box-sizing: border-box;">
  <div style="font-size: 15px; font-weight: 600; letter-spacing: 3px; text-transform: uppercase; color: #ffb300; margin-bottom: 8px;">Chapter 08</div>
  <div style="font-size: 36px; font-weight: 800; color: #ffffff; margin-bottom: 16px; line-height: 1.2;">High-Resolution Image Synthesis</div>
  <div style="width: 60px; height: 4px; background: linear-gradient(90deg, #ffb300, rgba(255,179,0,0.5)); border-radius: 2px; margin-bottom: 18px;"></div>
  <div style="font-size: 17px; color: #b0bec5; line-height: 1.6;">Autoregressive Image Generation, VQGAN & Discrete Tokenization</div>
  <div style="margin-top: 20px; font-size: 13px; color: #546e7a;">Source: ByteByteGo — Generative AI System Design Interview</div>
</div>

<div style="border-left: 5px solid #ff8f00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,143,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Introduction</span>
</div>


Generative AI offers a fascinating ability to create highly realistic and diverse images. In this chapter, we explore a technique that enables the generation of detailed and varied images in just seconds.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-1-7SJ24SOG.webp" alt="Figure 1: An image generated by VQGAN [1]" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 1: An image generated by VQGAN [1]</p>
</div>



<div style="border-left: 5px solid #ff8f00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,143,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Clarifying Requirements</span>
</div>


Here is a typical interaction between a candidate and an interviewer:


<div style="border: 1px solid rgba(255,179,0,0.3); border-radius: 8px; margin: 28px 0; overflow: hidden; font-family: system-ui, -apple-system, sans-serif;">
  <div style="background: rgba(255,179,0,0.08); padding: 10px 20px; border-bottom: 1px solid rgba(255,179,0,0.3); display: flex; align-items: center; gap: 8px;">
    <span style="font-size: 1.1em;">&#128483;</span>
    <span style="font-size: 0.85em; font-weight: 700; color: #ffb300; text-transform: uppercase; letter-spacing: 1.5px;">Interview Transcript</span>
  </div>
  <div style="padding: 24px 20px 24px 28px; position: relative;">
    <div style="position: absolute; left: 38px; top: 24px; bottom: 24px; width: 2px; background: rgba(255,179,0,0.3);"></div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffb300; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffb300; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffb300; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffb300; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Should the system focus on specific categories of images at the start?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">For simplicity, let's begin with natural scenes and urban landscapes. We can explore other categories later.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffb300; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffb300; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffb300; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffb300; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Do we have training data consisting of natural scenes? What's the dataset size?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">We have a large dataset with about 5 million high-resolution images of natural scenes and landscapes.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffb300; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffb300; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffb300; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffb300; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Should the system support additional conditioning, such as input text describing the desired image?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Good question. We'll focus on image generation without input conditions. However, the system should be flexible to support input prompts.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffb300; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffb300; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffb300; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffb300; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">What resolution range should we aim for when generating the images?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">The system should generate images with either 1024x1024 or 2048x2048 pixels, based on user requests.</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 22px; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #ffb300; border: 3px solid transparent; box-shadow: 0 0 0 2px #ffb300; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #ffb300; font-size: 0.95em;">Candidate</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #ffb300; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Candidate</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Should the images be generated in real time, or is some delay acceptable?</div>
      </div>
    </div>
    <div style="display: flex; gap: 18px; align-items: flex-start; margin-bottom: 0; position: relative;">
      <div style="width: 22px; height: 22px; border-radius: 50%; background: #8899aa; border: 3px solid transparent; box-shadow: 0 0 0 2px #8899aa; flex-shrink: 0; margin-top: 2px; z-index: 1;"></div>
      <div style="flex: 1;">
        <div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 4px;">
          <span style="font-weight: 700; color: #8899aa; font-size: 0.95em;">Interviewer</span>
          <span style="font-size: 0.75em; font-weight: 600; color: white; background: #8899aa; padding: 1px 8px; border-radius: 10px; text-transform: uppercase; letter-spacing: 0.5px;">Interviewer</span>
        </div>
        <div style="color: inherit; line-height: 1.7; font-size: 0.95em;">Real-time generation isn't necessary. However, a reasonable processing time is important. Let's aim for five seconds per image.</div>
      </div>
    </div>
  </div>
</div>


<div style="border-left: 5px solid #ff8f00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,143,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Frame the Problem as an ML Task</span>
</div>


<div style="border-bottom: 2px solid rgba(255,179,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffb300;">Specifying the system’s input and output</span>
</div>


For high-resolution image synthesis, the user simply requests a new image. The output is a high-resolution image.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-2-7URMKUHE.svg" alt="Figure 2: Input and output of an image generation system" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 2: Input and output of an image generation system</p>
</div>


<div style="border-bottom: 2px solid rgba(255,179,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffb300;">Choosing a suitable ML approach</span>
</div>


As discussed in Chapter 7, there are several approaches to image generation, including VAEs, GANs, autoregressive models, and diffusion models. In this section, we choose the one best suited for the task.

Most variants of VAEs and GANs struggle to generate high-resolution images, such as those with resolutions of 512x512 pixels and above. They face a challenge known as posterior collapse. This occurs because, as the resolution increases, these models require a decoder with a higher capacity to capture additional details. During training, the decoder can become so powerful that it starts to ignore input from the latent space, as it can model the output independently. As a result, the latent variables contribute little to the generation process, reducing the diversity of the images.

While both autoregressive and diffusion models can generate high-resolution images, they differ significantly in complexity and resource requirements. Autoregressive models are often considered slow due to their sequential nature, where each pixel depends on the ones generated before it. This dependency leads to a time complexity that increases linearly with the number of pixels, resulting in an $\mathcal{O}(N^2)$ complexity for an image of size $N \times N$, and the process is difficult to parallelize. To address this limitation, autoregressive models generate images chunk by chunk instead of pixel by pixel. For example, generating a 1024x1024 image using chunks of 64x64 pixels requires only 256 steps or tokens, significantly reducing the computational overhead compared to traditional pixel-based methods.

On the other hand, the complexity of diffusion models increases super-linearly with image size, resulting in a computational complexity of $\mathcal{O}(T N^2)$, where $N$ is the number of pixels and $T$ represents the number of denoising steps. Larger images often require more refinement steps to maintain quality and coherence, which further escalates computational demand.

In practice, generating a high-resolution image using standard diffusion models can take several minutes. In contrast, Transformer-based autoregressive models can accomplish similar tasks in seconds due to their chunk-based generation approach. For this chapter, we focus on autoregressive models for educational purposes. In Chapter 9, we will cover diffusion models in detail. Let's now delve into autoregressive models and their key components.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-3-G72EMMUJ.svg" alt="Figure 3: Autoregressive image generation" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 3: Autoregressive image generation</p>
</div>


Autoregressive models generate images by treating them as a sequence generation task. This approach relies on two primary components:


* Image tokenizer
* Image generator


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Image tokenizer</span>
</div>


Image tokenization refers to representing an image with a sequence of discrete tokens. This is crucial in autoregressive models, where the image is generated sequentially, chunk by chunk.

The image tokenizer is a separate model, trained independently. Its main functions are to encode an image into a sequence of discrete tokens and decode a sequence of discrete tokens back into an image.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-4-63PQYD35.svg" alt="Figure 4: Image tokenizer’s encoding and decoding" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 4: Image tokenizer’s encoding and decoding</p>
</div>


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Image generator</span>
</div>


The image generator is the primary model for generating images chunk by chunk. While there are various architectures for sequence generation, the decoder-only Transformer is the most effective choice for two reasons. First, the decoder-only Transformer has a flexible architecture that can handle different modalities. In a chatbot, it takes text tokens as input and generates text tokens as output. In image captioning, it takes an image as input and outputs text tokens. For image generation, it generates a sequence of image tokens as output, which are then decoded into an image.

Second, the Transformer architecture is effective at capturing long-range dependencies through its attention mechanism, which is beneficial for generating coherent images.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-5-7BZVHBUQ.svg" alt="Figure 5: Decoder-only Transformer’s flexibility in handling various modalities" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 5: Decoder-only Transformer’s flexibility in handling various modalities</p>
</div>


In summary, we approach image generation with a Transformer-based autoregressive model. First, an image generator (decoder-only Transformer) generates a sequence of discrete tokens. Then, an image tokenizer decodes these tokens into the final image. We will explore the architecture, training, and sampling processes of these components in detail in the model development section.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-6-F4GZP66Z.svg" alt="Figure 6: Autoregressive image generation" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 6: Autoregressive image generation</p>
</div>


<div style="border-left: 5px solid #ff8f00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,143,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Data Preparation</span>
</div>


The data preparation process involves two crucial steps:


* Image cleaning and normalization
* Image tokenization


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Image cleaning and normalization</span>
</div>


In this step, we remove low-quality images from the training data and ensure the remaining ones are consistent. This is achieved by applying the following operations:


* Remove low-quality images: We remove images with low resolution, excessive noise, or irrelevant content. We also ensure the dataset includes a wide range of styles, subjects, and compositions. This step is crucial for the generative model to produce diverse, high-quality images.
* Normalize images: Normalization involves scaling pixel values to a range, typically 0 to 1, to stabilize the training process.
* Resize images: Images often come in different sizes and aspect ratios. Resizing them to a uniform size ensures the model receives consistent inputs. Based on the interviewer’s requirements, we resize all images to 1024x1024.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Image tokenization</span>
</div>


The image generator requires images to be represented as a sequence of discrete tokens. To achieve this, after training the image tokenizer, we tokenize all images in our training dataset into discrete tokens. It's important to note that this data preparation step is intended primarily for the image generator, not the image tokenizer.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-7-U4EVDN3X.svg" alt="Figure 7: Data preparation process" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 7: Data preparation process</p>
</div>


These two steps ensure the training data is high-quality, consistent, and represented as a sequence of numerical inputs.


<div style="border-left: 5px solid #ff8f00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,143,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Model Development</span>
</div>


<div style="border-bottom: 2px solid rgba(255,179,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffb300;">Architecture</span>
</div>


In this section, we explore the architecture of both the image tokenizer and the image generator.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Image tokenizer</span>
</div>


The image tokenizer model has two functions:


* 1. Encoding an image into a sequence of discrete tokens
* 2. Decoding a sequence of discrete tokens back into an image


A common architecture specifically designed for image tokenization is the Vector-Quantized VAE (VQ-VAE) [2], which is a variant of the standard VAE discussed in Chapter 7. The VQ-VAE consists of three components:


* Encoder
* Quantizer
* Decoder


<strong style="color: #ffb300;">Encoder</strong>


The encoder maps the input image into a lower-dimensional latent space. This component encodes important features of the image into an encoded representation.

The encoder's architecture is a deep convolutional neural network (CNN) with several convolution layers, each followed by a ReLU [3] activation function. These layers process the input image and extract visual features.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-8-HCJEHNBO.svg" alt="Figure 8: The encoder converts an input image into an encoded representation containing 9 features, each with c channels" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 8: The encoder converts an input image into an encoded representation containing 9 features, each with c channels</p>
</div>


<strong style="color: #ffb300;">Quantizer</strong>


The quantizer converts continuous latent vectors into discrete tokens. There are two main reasons why VQ-VAE introduces a quantizer component to a standard VAE:


* Avoiding posterior collapse
* Reducing the learning space


Avoiding posterior collapse: Posterior collapse is a common issue in standard VAEs where the latent variables contribute little or are ignored because the decoder generates accurate outputs without using the latent space. The quantization step addresses this by discretizing the latent variables, thus forcing the model to use them during reconstruction. This ensures the decoder doesn't overpower the latent space and keeps the latent variables actively involved in shaping the output.


Reducing the learning space: Continuous vectors are difficult to predict sequentially because they have endless possibilities and small differences. By turning these vectors into discrete tokens, the quantizer simplifies the process by allowing the Transformer to focus on fewer options.

The quantizer uses an internal codebook to convert continuous latent vectors into discrete tokens. This codebook contains learnable embeddings that represent different patterns in the input images. Each embedding acts as a token, represented by an integer from 1 to k. The quantizer replaces each continuous vector with the closest token in the codebook based on Euclidean distance [4].


Note that the quantizer is an embedding table. Its sole parameter is the codebook, which is learned during training. The quantizer’s single responsibility is to map each continuous vector with the closest token in the codebook; therefore, the output is a collection of token IDs.


<strong style="color: #ffb300;">Decoder</strong>


The decoder converts discrete tokens back into the original image. It typically uses a deep CNN with transposed convolutions (ConvTranspose2d) to gradually transform the representation to the original image size. To learn more about convolutions and transposed convolutions, refer to [5].


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-10-BLFPCVZB.svg" alt="Figure 10: Decoding process" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 10: Decoding process</p>
</div>


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Image generator</span>
</div>


The image generator generates a sequence of discrete tokens representing an image. As mentioned earlier, a decoder-only Transformer is often used for sequence generation tasks, which includes the following components:


* Embedding lookup: Replaces each discrete token with its embedding from the codebook.
* Projection: Projects each token embedding into a dimensionality that matches the Transformer's internal representation.
* Positional encoding: Adds positional encodings to the sequence to provide spatial information.
* Transformer: Processes the input sequence and outputs an updated sequence of vectors.
* Prediction head: Utilizes the updated embeddings to predict the next token.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-11-BIXTWXD2.svg" alt="Figure 11: Decoder-only Transformer components" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 11: Decoder-only Transformer components</p>
</div>


<div style="border-bottom: 2px solid rgba(255,179,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffb300;">Training</span>
</div>


In autoregressive image generation, we have two training stages:


* Stage I: Training the image tokenizer
* Stage II: Training the image generator


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Stage I: Training the image tokenizer</span>
</div>


The training process involves optimizing the encoder, decoder, and codebook so the model can accurately reconstruct the original images. This process can be described in three steps:


* 1. The encoder processes an input image and converts it into a continuous representation.
* 2. The quantizer replaces the continuous representation with discrete tokens using its internal codebook.
* 3. The decoder uses the discrete tokens to reconstruct the original image.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-12-ANHE7F3R.svg" alt="Figure 12: Image tokenizer training process" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 12: Image tokenizer training process</p>
</div>


Since the quantizer lookup operation lacks a well-defined gradient for backpropagation, the VQ-VAE paper proposes approximating the gradient by copying it from the decoder input directly to the encoder output. This approach means that only the selected tokens receive gradients from the decoder, while unselected tokens do not receive any gradients.


<strong style="color: #ffb300;">Training data</strong>


We train the image tokenizer with 5 million images. Since the training is self-supervised and doesn't require image labels, we include other publicly available image datasets to enhance the tokenizer's robustness. In particular, we use the LAION-400M dataset [6], which contains 400 million images. This results in a richer codebook that captures diverse visual patterns.


<strong style="color: #ffb300;">ML objective and loss function</strong>


The ML objective of the image tokenizer is to accurately reconstruct original images from their quantized tokens. To achieve this ML objective, the following loss functions are typically employed during the training process:


* Reconstruction loss
* Quantization loss


Reconstruction loss: The reconstruction loss measures the difference between the original image and its reconstruction from the quantized tokens. It is typically calculated using the mean squared error (MSE) formula:


$$ \text{Reconstruction loss} = \frac{1}{n} \sum_{i=1}^n\left(x_i-\hat{x}_i\right)^2 $$


Where:


* $x_i$ is the pixel value of the original image,
* $\hat{x}_i$ is the pixel value of the reconstructed image,
* $n$ is the total number of pixels in the image.


Quantization loss: The quantization loss measures the distance between the encoder’s outputs and the nearest embedding in the codebook. This loss encourages the encoder to produce outputs that are closer to the codebook embeddings.


$$ \text{Quantization loss} = \left\|\operatorname{sg}[E(x)]-z_q\right\|_2^2+\left\|\operatorname{sg}\left[z_q\right]-E(x)\right\|_2^2 $$


Where:


* $E(x)$ is the continuous latent vector produced by the encoder, $E$, from the input $x$,
* $z_q$ is the quantized latent vector selected from the codebook $Z$,
* $\operatorname{sg}(.)$ represents the stop-gradient operation that blocks the gradients from flowing through the term. It is used here to prevent the codebook from being updated when optimizing the encoder.


For more details on the quantization loss formula, refer to the VQGAN paper [1].


In practice, using both reconstruction loss and quantization loss during training works well for reconstructing low-resolution images. However, for high-resolution images, the model may still produce artifacts. To improve reconstruction quality at high resolutions, two additional loss functions are typically employed:


* Perceptual loss
* Adversarial loss


Perceptual loss: Perceptual loss measures the difference between the features of the original and reconstructed images extracted from a specific layer of a pretrained model such as VGG [7]. The formula is:


$$ \text{Perceptual loss} = \sum_l\left\|\phi_l(x)-\phi_l(\hat{x})\right\|_2^2 $$


Where:


* $\phi_l$ denotes the feature map of the layer, $l$, from a pretrained VGG model,
* $x$ is the original image,
* $\hat{x}$ is the reconstructed image.


The perceptual loss encourages the model to reconstruct images that are perceptually similar to the original images. VGG features encode high-level details such as content and style. The perception loss guides the training process so that the model can better preserve these details in the reconstructed images.


Adversarial loss: Adversarial loss is derived from GANs [8], where a discriminator tries to distinguish between real and reconstructed images. This loss is used to measure how well the image reconstructed by the image tokenizer can fool the discriminator. The formula, as we saw in Chapter 7, is:


$$ \text{Adversarial loss} = -\log (D(\hat{x})) $$


Where:


* $D$ is the discriminator network,
* $\hat{x}$ is the reconstructed image.


This loss function encourages the model to produce reconstructed images that a trained discriminator cannot distinguish from real images. The VQGAN paper introduced a patch-based version of this loss to reduce unnatural artifacts and improve the realism of the reconstructions.


Overall loss: The overall loss function is often a weighted sum of the individual losses described above. The weights ($\lambda_i$) are hyperparameters that need tuning based on specific performance goals and experiments.


$$ \begin{aligned} \text { Overall loss }= \lambda_{\text {rec }} & \times \text { reconstruction loss }+ \\ \lambda_{\text {quant }} &\times \text { quantization loss }+ \\ \lambda_{\text {perc }} &\times \text { perceptual loss }+ \\ \lambda_{\text {adv }} &\times \text { adversarial loss } \end{aligned} $$


After training the image tokenizer, we convert all 5 million training images into discrete tokens and cache them, as detailed in the data preparation section. This step ensures that all images are represented as a sequence of discrete tokens, which is required for training the image generator.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Stage II: Training the image generator</span>
</div>


Training the image generator, which is a decoder-only Transformer, is similar to the process described in earlier chapters. The training data consists of sequences of discrete tokens, and the model learns to predict these tokens sequentially during training.

We employ next-token prediction as our ML objective and cross-entropy as the loss function to measure how accurate the predicted probabilities are compared to the correct visual tokens.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-13-TYIGS3YR.svg" alt="Figure 13: Image generator loss calculation" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 13: Image generator loss calculation</p>
</div>


<div style="border-bottom: 2px solid rgba(255,179,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffb300;">Sampling</span>
</div>


In autoregressive models, generating a new image involves two steps:


* 1. Generating a sequence of discrete tokens
* 2. Decoding discrete tokens into an image


<strong style="color: #ffb300;">1. Generating a sequence of discrete tokens</strong>


In the first step, the image generator produces a sequence of tokens. The autoregressive nature of the generation ensures that each token is conditioned on preceding tokens, leading to coherent images.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-14-ETCOCZBV.svg" alt="Figure 14: Generating a sequence of discrete tokens using the image generator" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 14: Generating a sequence of discrete tokens using the image generator</p>
</div>


Here is a step-by-step process to autoregressively generate a sequence of tokens:


* 1. Randomly select a token from the codebook as the first token. This initial token acts as a seed for the rest of the generation process.
* 2. Autoregressively generate tokens one by one. This involves:
*     1. Passing the current sequence of tokens to the image generator to predict the probability distribution over the codebook
*     2. Selecting the next token using a sampling method such as top-p sampling
*     3. Appending the chosen token to the current sequence


This process continues until the entire image is generated. The number of iterations depends on the resolution and size of the desired output image. For example, generating an image of 1024x1024 pixels, with each visual token representing a 64x64 pixel block, requires 256 tokens. The process continues until all 256 tokens are generated. Once the sequence of tokens is complete, it is transformed into an actual image, which is the focus of the next step.


<strong style="color: #ffb300;">2. Decoding discrete tokens into an image</strong>


In this step, the sequence of discrete tokens is transformed into an image by using the decoding functionality of the image tokenizer.


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-15-63IZAYFF.svg" alt="Figure 15: Decoding tokens into an image" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 15: Decoding tokens into an image</p>
</div>


<div style="border-left: 5px solid #ff8f00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,143,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Evaluation</span>
</div>


The evaluation metrics for high-resolution image synthesis are similar to those in Chapter 7. We'll briefly review them in this section without going into detail.


<div style="border-bottom: 2px solid rgba(255,179,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffb300;">Offline evaluation metrics</span>
</div>


The following metrics are typically employed to measure the quality and diversity of the generated images:


* Inception score: Measures how similar the generated images are to images of real-world objects by utilizing a pretrained Inception v3 model. To learn more about the Inception score, refer to [9].
* Fréchet inception distance (FID): Compares the distribution of generated images to real images by comparing features extracted from a pretrained Inception v3 model. This metric measures how similar the statistics of generated and real images are. To learn more about FID, refer to [10].
* Human evaluation: Human evaluators are presented with pairs of images and asked to judge their photorealism and aesthetic qualities. The votes provide a statistical measure of which models produce more realistic images over time.


In addition to those metrics, it is common to evaluate other aspects of the model such as latency and cost.


* Time to generate an image: Measures the time it takes for the model to generate an image. This metric is important to monitor since users generally expect quick results.
* Cost per generation: Calculates the cost to generate an image. This metric depends on factors such as model complexity, resolution, and infrastructure expenses. Monitoring the cost of generation is crucial as it impacts business revenue.


<div style="border-bottom: 2px solid rgba(255,179,0,0.3); padding-bottom: 8px; margin: 28px 0 14px 0;">
  <span style="font-size: 1.3em; font-weight: 600; color: #ffb300;">Online evaluation metrics</span>
</div>


In practice, companies monitor various metrics to assess the system's real-time quality. Common metrics include:


* User feedback: Collects direct feedback from users regarding generated images.
* Periodic surveys: Gathers user opinions on the quality and relevance of generated images.
* Subscription rate: Measures how often users subscribe to services or features related to image generation.
* Churn rate: Measures the rate at which users stop using the service.


<div style="border-left: 5px solid #ff8f00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,143,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Overall ML System Design</span>
</div>


Once we are satisfied with the performance of the image generator and image tokenizer models, we can integrate them to construct the image synthesis system. The primary components in a high-resolution image synthesis system are:


* Generation service
* Decoding service
* Super-resolution service


<div style="max-width: 80%; margin: 20px auto; box-sizing: border-box;">
<img src="assets/figure-8-16-RA4BP4II.svg" alt="Figure 16: High-resolution image synthesis ML design" style="max-width: 100%; height: auto; display: block; border-radius: 8px; box-shadow: 0 2px 12px rgba(0,0,0,0.15); background: white;" />
<p style="text-align: center; font-size: 0.92em; color: #90a4ae; margin-top: 8px; font-style: italic;">Figure 16: High-resolution image synthesis ML design</p>
</div>


Understanding the purpose of each component and their interactions will provide a holistic view of the system. Let’s explore each in more detail.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Generation service</span>
</div>


The generation service handles user requests and interacts with the trained image generator model to produce a sequence of visual tokens.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Decoding service</span>
</div>


The decoding service interacts with the image tokenizer to convert the generated sequence of visual tokens into an image. Note that when we deploy the model, we don’t need the encoder in the image tokenizer – it is only used during training.

Separating generation and decoding services is crucial because the image generator and tokenizer are different models with distinct computational needs and latencies. This approach allows each service to scale independently and manage resources efficiently.


<div style="margin: 22px 0 10px 0; border-left: 3px solid #ffb300; padding-left: 10px;">
  <span style="font-size: 1.15em; font-weight: 600; font-style: italic; color: #ffb300;">Super-resolution service</span>
</div>


Super-resolution service uses a pretrained model to increase the resolution of generated images. For example, if the desired resolution is 2048x2048 but the generator produces only 1024x1024, we use a super-resolution model with a 2x upscale factor.

This service is crucial for applications requiring detailed and realistic visuals, such as medical imaging. There are many established solutions for super-resolution, from CNN-based [11] to GAN-improved [12]. To learn more about recent approaches, refer to [13].


<div style="border-left: 5px solid #ff8f00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,143,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Other Talking Points</span>
</div>


If there's time remaining at the end of the interview, you could explore these additional points:


* Extending autoregressive models to support text-based generation [14] [15].
* Support applications such as image completion and image super-resolution [16].
* Balancing diversity vs. fidelity in sampling, using techniques such as temperature scaling [17].
* Enhancing the stability with adversarial training, gradient clipping, and learning rate scheduling [18][19].
* Using progressive growing and multi-scale architectures to improve image quality and detail [20].
* Creating interactive systems for users to refine and customize generated images [21].


<div style="border-left: 5px solid #ff8f00; padding: 12px 0 12px 20px; margin: 35px 0 18px 0; background: linear-gradient(90deg, rgba(255,143,0,0.08) 0%, transparent 60%); border-radius: 4px;">
  <span style="font-size: 1.6em; font-weight: 700;">Reference Material</span>
</div>


[1] Taming Transformers for High-Resolution Image Synthesis. https://arxiv.org/abs/2012.09841.

[2] Neural Discrete Representation Learning. https://arxiv.org/abs/1711.00937.

[3] Deep Learning using Rectified Linear Units (ReLU). https://arxiv.org/abs/1803.08375.

[4] Euclidean distance. https://en.wikipedia.org/wiki/Euclidean_distance.

[5] A guide to convolution arithmetic for deep learning. https://arxiv.org/abs/1603.07285.

[6] LAION data set 400 million https://laion.ai/blog/laion-400-open-dataset/.

[7] Very Deep Convolutional Networks for Large-Scale Image Recognition. https://arxiv.org/abs/1409.1556.

[8] Generative Adversarial Networks. https://arxiv.org/abs/1406.2661.

[9] Inception score. https://en.wikipedia.org/wiki/Inception_score.

[10] FID calculation. https://en.wikipedia.org/wiki/Fr%C3%A9chet_inception_distance.

[11] Image Super-Resolution Using Very Deep Residual Channel Attention Networks. https://arxiv.org/abs/1807.02758.

[12] ESRGAN: Enhanced Super-Resolution Generative Adversarial Networks. https://arxiv.org/abs/1809.00219.

[13] NTIRE 2024 Challenge on Image Super-Resolution (x4): Methods and Results. https://arxiv.org/abs/2404.09790.

[14] Muse: Text-To-Image Generation via Masked Generative Transformers. https://arxiv.org/abs/2301.00704.

[15] VQGAN-CLIP: Open Domain Image Generation and Editing with Natural Language Guidance. https://arxiv.org/abs/2204.08583.

[16] LAR-SR: A Local Autoregressive Model for Image Super-Resolution. https://openaccess.thecvf.com/content/CVPR2022/papers/ Guo_LAR-SR_A_Local_Autoregressive_Model_for_Image_Super-Resolution_CVPR_2022_paper.pdf.

[17] Long Horizon Temperature Scaling. https://arxiv.org/abs/2302.03686.

[18] Learning Rate Scheduling. https://d2l.ai/chapter_optimization/lr-scheduler.html.

[19] Adversarial Training. https://adversarial-ml-tutorial.org/adversarial_training/.

[20] Progressive Growing of GANs for Improved Quality, Stability, and Variation. https://arxiv.org/abs/1710.10196.

[21] CogView2: Faster and Better Text-to-Image Generation via Hierarchical Transformers. https://arxiv.org/abs/2204.14217.
